# 03 先设计验证 再解读真实结果

合成实验知道真值，实景首先需要明确**比较的是同一块地、同一种物理量和同一批有效样本吗？** 

本页分别标注各条实验分支。2013 年 AVIRIS/MODIS 用当前影像背景模型，回答场景内的端元与尺度问题；2016/2017 年历史库谱结果只重算归档指标；新增公开库谱试验则从同一批 Landsat B2–B7 观测重新解混，以公开 WorldView 派生雪覆盖作独立参照。新增同日 MODIS 与 WorldView 公开派生参考的对照，评价另一个公开端元模型。各项结果都对应自己的数据和方法。

首行会按数据打印 62,908 个有效 30 m 像元、420 条 ROI 单元记录、406 个原生 1 km 单元。图件包含越界系数状态、FCLS 比例、留出像元扫描、雪谱、面积与 QA 筛选、独立覆盖比较和尺度分析。

新增同日 MODIS 实验中，2016 年严格七波段共同域有 1,416 个 500 m 单元，其中 463 个参考雪比例≥0.1；独立状态标志对照为 408 个 1 km 单元，含 123 个阳性参考。2017 年阳性较少，单独报告其分层结果。

In [ ]:
%matplotlib inline
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from snowmix import teaching as t
from snowmix.case_analysis import absorption_area, binary_agreement
from snowmix.diagnostics import coefficient_status
from snowmix.validation_study import independent_transfer

case = t.load_aviris_case()
diagnostic = t.load_coefficient_diagnostics()
demo = t.demo_directory()
units = pd.read_csv(demo / "modis_1km_units.csv")
native_units = t.unique_native_units(units)
spatial = json.loads((demo / "spatial_metadata.json").read_text(encoding="utf-8"))
print(f"教学数据：{int(case['valid_mask'].sum()):,} 个有效 30 m 像元；{len(units)} 条 ROI 记录；{len(native_units)} 个原生 1 km 单元")
print("绘图标签：" + ("中文" if t.configure_plots() else "English（未安装支持的中文字体）"))
assert case["snow_spectra"].shape == (108, 182)
assert case["backgrounds"].shape == (182, 3)
assert diagnostic["fractions"].shape == (3, 62908, 4)
assert len(native_units) == 406

## 1 先判断哪些结果具有面积比例的含义

MGS 是修正 Gram–Schmidt（modified Gram–Schmidt）正交化最小二乘，保留原文允许的有符号系数。FCLS 是全约束最小二乘（fully constrained least squares），要求各端元系数非负、总和不超过 1，阴影为剩余部分。输入的 4 个端元依次为雪、植被、亮裸地和暗裸地；它们表示预选的光谱类型。

原始系数用于重建光谱。**去阴影雪比例（normalized snow fraction）**定义为雪系数除以全部地物系数之和，即 $F_s/\sum_i F_i$。分母不超过 0.05 时留空。该归一化不会把无约束解自动变成物理面积：还需检查雪、其他地物和阴影的每个系数都在[0,1]内。

下面在相同 62,861 个可归一化像元上，分别计算雪系数越界率和全部系数可行率。储存为 float32 的系数用 $10^{-7}$ 容差判范围。图中有符号 MGS 使用发散色标；状态图给越界区域单独着色；右上 FCLS 图才用 0–1 比例色标。

In [ ]:
common = np.isfinite(diagnostic["normalized"]).all(axis=(0, 2))
assert common.sum() == 62861
rows = []
for i, name in enumerate(diagnostic["method_names"]):
    status = coefficient_status(diagnostic["fractions"][i], diagnostic["shade"][i], tolerance=1e-7)
    normalized = diagnostic["normalized"][i, common, 0]
    rows.append({"method": name, "n": int(common.sum()),
        "negative_percent": 100*status["negative_snow"][common].mean(),
        "above_percent": 100*np.mean(normalized > 1+1e-7),
        "all_feasible_percent": 100*status["feasible"][common].mean(),
        "rms_pp": 100*diagnostic["rms"][i, common].mean()})
physical_table = pd.DataFrame(rows)
display(t.readable_table(physical_table, {"method":"模型", "n":"共同像元",
    "negative_percent":"负雪系数 / %", "above_percent":"雪比例>1 / %",
    "all_feasible_percent":"全部系数可行 / %", "rms_pp":"光谱 RMS / pp"}))
assert physical_table.loc[physical_table.method=="physical_FCLS", "all_feasible_percent"].iloc[0] == 100
for i, key in enumerate(("physical_snow_fraction", "image_snow_fraction", "fcls_snow_fraction")):
    np.testing.assert_allclose(diagnostic["normalized"][i,:,0], case[key][case["valid_mask"]],
        atol=2e-6, rtol=2e-6, equal_nan=True)
figure = t.plot_coefficient_status(diagnostic, spatial)
display(figure); plt.close(figure)

负雪系数不是浮点舍入造成的小量。它们主要出现在高植被指数及其他背景光谱中，模型通过正负系数的组合补偿谱形差异。下面按预设 NDSI/NDVI 规则描述这些位置；这些分组只基于影像光谱，未充当实测分类真值。单条植被和亮/暗裸地谱对场景的代表性有限，照明与几何近似也可能影响拟合；区分各机制还需要额外观测。

In [ ]:
groups = pd.read_csv(demo / "coefficient_spectral_groups.csv")
display(t.readable_table(groups, {"method":"模型", "spectral_group":"光谱分组",
    "n":"像元数", "negative_fraction":"负雪系数比例", "normalized_snow_mean":"去阴影雪比例均值",
    "rms_mean":"光谱 RMS"}))
support = pd.read_csv(demo / "sample_support.csv")
branch_names = {"historical_roi":"历史库谱输入", "image_background_input":"当前影像背景输入",
    "fixed_five_common":"五半径共同域", "three_methods_common":"三模型共同域",
    "spectral_degradation_common":"历史仅光谱对照"}
support["branch"] = support.branch.map(branch_names)
display(t.readable_table(support, {"branch":"统计分支", "n":"像元数", "condition":"筛选条件"}, precision=0))

63,213 个历史输入与 62,908 个当前输入的差别，是新增原生有效覆盖率≥95% 的筛选。60,949、62,861 和 62,870 分别属于不同实验的共同域，不能接成一条依次减少的链条。每张表都在自己的共同域内比较。

## 2 一次只改雪粒半径

固定观测、背景、182 个波段、太阳角和求解器，只将雪半径设为 50、100、200、500、1000 μm。原先的四条示例中有三条正是背景端元本身：用同一基底拟合当然得到近零雪系数和残差。它们可作数值自拟合控制，却不能代表普通无雪像元的精度。

这里改用**留出光谱**作主要图示。选样预先依据高 NDSI、中等 NDSI、高 NDVI 和低 NDSI 裸地分组的中位亮度，排除全部端元周围 3 个像元；选择时不看 RMS 或 WorldView 参照。输入形状为 4 像元×182 波段，输出系数形状为 5 半径×4 像元×4 端元。

In [ ]:
with np.load(demo / "heldout_spectra.npz", allow_pickle=False) as archive:
    heldout = dict(archive)
heldout_case = dict(case, representative_spectra=heldout["spectra"],
    representative_row_col=heldout["row_column"], heldout=True)
scan = t.scan_representatives(heldout_case)
np.testing.assert_allclose(scan["rms"], np.sqrt(np.mean(scan["residual"]**2, axis=2)), atol=1e-12)
r, c = heldout["row_column"].T
np.testing.assert_allclose(scan["normalized"][:,:,0], case["fixed_snow_fraction"][:,r,c],
    rtol=2e-5, atol=2e-6, equal_nan=True)
representatives = t.representative_table(scan)
display(t.readable_table(pd.DataFrame({"编号":np.arange(4), "行":r, "列":c,
    "NDSI":heldout["ndsi"], "NDVI":heldout["ndvi"]})))
display(t.readable_table(representatives, {"representative":"像元编号", "radius_um":"半径 / μm",
    "raw_snow":"原始雪系数", "normalized_snow":"去阴影雪比例", "rms_reflectance":"光谱 RMS"}, precision=4))
figure = t.plot_representative_scan(heldout_case, scan)
display(figure); plt.close(figure)

### 从背景正交补看雪系数的负号

为了理解负号，可以先移除背景张成的光谱子空间。令 B 为三条背景组成的矩阵，u=s−BB⁺s 为雪谱在背景子空间外的部分，r 为观测扣除背景投影后的残差。雪系数满足
$$F_s=\frac{u^\mathsf{T}r}{u^\mathsf{T}u}.$$
如果观测残差在该雪方向上的投影为负，无约束最小二乘就返回负雪系数。下面用同一组四条留出观测计算分子、分母，再与固定 200 μm 的 MGS 解核对。它解释了求解器怎样产生负号；背景代表性和观测几何等因素在讨论中分别分析。

In [ ]:
from snowmix.linear import mgs_operator
B = case["backgrounds"]  # 182波段 × 3背景，B_plus 为3 × 182。
B_plus = mgs_operator(B)
snow200 = case["snow_spectra"][np.flatnonzero(case["radii_um"]==200)[0]]
direction = snow200 - B @ (B_plus @ snow200)
residual_to_background = heldout["spectra"] - (heldout["spectra"] @ B_plus.T) @ B.T
numerator = residual_to_background @ direction
denominator = direction @ direction
snow_from_projection = numerator / denominator
fixed_index = int(np.flatnonzero(scan["radii_um"]==200)[0])
np.testing.assert_allclose(snow_from_projection, scan["fractions"][fixed_index,:,0], atol=1e-12)
display(t.readable_table(pd.DataFrame({"像元编号":np.arange(4), "投影分子":numerator,
    "方向平方范数":denominator, "投影所得雪系数":snow_from_projection,
    "MGS雪系数":scan["fractions"][fixed_index,:,0]}), precision=5))

图中的空白比例表示分母≤0.05，按规则记为 NaN；它不是“没有雪”的标签。曲线保留实际弯曲与拐点。原始雪系数与去阴影比例回答不同问题，不能用两个全区均值相除来反推平均分母：每个像元的分母不同，且与系数共同变化。

全区比较还需固定样本。五个半径共用 60,949 个像元，直接由原始系数和得到分母；所有共同像元的分母均超过 0.05。接着用 30 个 1.5 km 空间区块做 1,000 次配对 bootstrap，边缘区块按像元数加权；每次所有模型抽相同区块，最后逐次计算方法差。

In [ ]:
fixed_statistics, paired = t.fixed_radius_statistics(case, repeats=1000, seed=1998)
frozen = pd.read_csv(demo / "fixed_radius_uncertainty.csv")
np.testing.assert_allclose(fixed_statistics, frozen, rtol=1e-6, atol=1e-8)
assert paired["spatial_blocks"] == 30
table = fixed_statistics.copy()
table["snow_ci"] = [f"{100*m:.2f} [{100*a:.2f}, {100*b:.2f}]" for m,a,b in zip(
    table.normalized_snow_mean, table.normalized_snow_ci_low, table.normalized_snow_ci_high)]
table["rms_ci"] = [f"{100*m:.3f} [{100*a:.3f}, {100*b:.3f}]" for m,a,b in zip(
    table.rms_mean, table.rms_ci_low, table.rms_ci_high)]
display(t.readable_table(table, {"radius_um":"半径 / μm", "common_n":"共同像元",
    "snow_ci":"去阴影雪比例 / % [95%区间]", "rms_ci":"RMS / pp [95%区间]"}))
table["input_pass_percent"] = 100*table.denominator_pass_fraction_input
display(t.readable_table(table, {"radius_um":"半径 / μm", "denominator_mean":"分母均值",
    "denominator_p05":"分母P05", "denominator_p50":"分母P50", "input_pass_percent":"全输入达标 / %"}))
delta = pd.DataFrame([{"指标":name, "均值差 / pp":100*value["mean"],
    "95%下限 / pp":100*value["interval_95"][0], "95%上限 / pp":100*value["interval_95"][1]}
    for name,value in paired["paired_delta"].items()])
display(t.readable_table(delta))
assert (fixed_statistics.denominator_pass_fraction_common == 1).all()

这些区间量化本场景、固定端元和掩膜下的空间抽样变化。1000−50 μm 的 RMS 均值差约−1.104 pp，去阴影雪比例差约−10.20 pp；这是对拟合及估计值变化的证据。覆盖是否更准确交给独立参考评价。

## 3 用影像雪谱检查谱库选择

物理雪库控制了半径，影像雪谱包含现场照明与粗糙度。为检查谱库选择，我先固定 NDSI>0.7、|NDVI|<0.15、绿光反射率 0.65–1.1、近红外反射率>0.11 的筛选规则，再拟合。

在 950–1090 nm 以两端连线作连续统 C，积分 $1-R/C$，取吸收面积 5%、25%、50%、75%、95% 分位附近的不同像元。输出单位为μm，五条曲线的编号表示不同谱型。原文用 Nolin 吸收积分标定粒径；这里没有该标定，只将面积作为选谱指标。

In [ ]:
area = absorption_area(case["wavelength_um"], case["image_snow_spectra"])
np.testing.assert_allclose(area, case["image_absorption_area_um"], atol=1e-12)
fig, ax = plt.subplots(figsize=(8.6, 4))
for i,(spectrum,color) in enumerate(zip(case["image_snow_spectra"], t.COLORS)):
    t.plot_spectrum(ax, case["wavelength_um"], spectrum, color=color, label=f"S{i+1}: A={area[i]:.4f} μm")
ax.set(xlabel=t.label("波长 / μm","Wavelength / μm"), ylabel=t.label("反射率","Reflectance"))
ax.legend(fontsize=8); fig.tight_layout(); display(fig); plt.close(fig)
display(t.readable_table(pd.DataFrame({"谱编号":np.arange(1,6),
    "行":case["image_endmember_row_col"][:,0], "列":case["image_endmember_row_col"][:,1],
    "吸收面积 / μm":area}), precision=5))

## 4 验证设计 从原生空间单元检查面积与 QA

MOD09GA 的 bit12 是 MOD35 snow/ice 标志，bit15 是 internal snow mask。它们属于原生约 1 km 状态网格，不把复制到 500 m 的值作为四个独立参照。

还需处理 ROI 重叠：原表有 420 条“ROI+网格行列”记录，其中 14 个原生边缘单元出现两次，合并后为 406 个空间单元。重叠处按有效面积较大的记录保留，并核对状态一致；选择不使用雪比例或 RMS。100 个面积合格单元和最终 17 个共同有效单元的数量不变。

图中先看 bit15 阳性落在哪里，再看 AVIRIS 面积、MODIS 输出面积和排除原因。MODIS 输出面积同时受影像质量和模型条件影响。状态 QA 单列检查，以区分“雪标志对照”与“两套反射率解混结果对照”。历史 85 个 500 m 配对属于另一个实验分支。

In [ ]:
figure = t.plot_modis_support(units, spatial)
display(figure); plt.close(figure)
ledger = t.validation_ledger(units)
np.testing.assert_array_equal(ledger.n, [406,406,100,17])
np.testing.assert_array_equal(ledger.flag12_positive, [31,31,0,0])
np.testing.assert_array_equal(ledger.flag15_positive, [116,116,20,0])
display(t.readable_table(ledger, {"stage":"筛选阶段", "n":"原生单元",
    "flag12_positive":"bit12阳性", "flag15_positive":"bit15阳性"}, precision=0))

In [ ]:
figure = t.plot_validation_design(ledger)
display(figure); plt.close(figure)
sensitivity = t.coverage_sensitivity(units)
pd.testing.assert_frame_equal(sensitivity, pd.read_csv(demo/"coverage_sensitivity.csv"), check_dtype=False)
display(t.readable_table(sensitivity, {"minimum_coverage":"面积门槛", "stage":"筛选阶段",
    "n":"单元数", "flag12_positive":"bit12阳性", "flag15_positive":"bit15阳性"}, precision=2))
state_ledger = t.state_flag_ledger(units)
display(t.readable_table(state_ledger, {"stage":"状态QA阶段", "n":"单元数",
    "flag12_positive":"bit12阳性", "flag15_positive":"bit15阳性"}, precision=0))
np.testing.assert_array_equal(state_ledger.n, [100,55,17])
np.testing.assert_array_equal(state_ledger.flag15_positive, [20,10,0])

三档面积门槛都得到 17 个共同阴性单元。进一步按状态位分解，100 个 AVIRIS 面积合格单元中有 20 个 bit15 阳性；晴空、无云影/卷云/内部云后剩 10 个；这 10 个全部带邻云标记。因而只放宽面积门槛不能补回阳性，继续放开邻云则会引入不同的质量条件。原有严格样本保留用于阴性一致性，雪区评价需另取有充分共同覆盖和清晰观测的场景。

### 用一个明确的合成例子学习混淆矩阵

下面 8 个值专用于教学。以 0.5 将比例转成预测类别，手算得到 TP=2、FN=2、FP=1、TN=3；召回率 0.5，特异度 0.75。先预测改变阈值会使哪几个样本换类，再运行。

In [ ]:
synthetic_flag = np.array([1,1,1,1,0,0,0,0])
synthetic_fraction = np.array([.9,.7,.2,.05,.8,.4,.1,0])
synthetic = binary_agreement(synthetic_flag, synthetic_fraction, threshold=.5)
assert (synthetic["tp"],synthetic["fn"],synthetic["fp"],synthetic["tn"]) == (2,2,1,3)
assert synthetic["recall"] == .5 and synthetic["specificity"] == .75
display(pd.DataFrame([[2,2],[1,3]], index=["参照有雪","参照无雪"], columns=["预测有雪","预测无雪"]))
agreement = t.product_consistency(units)
display(t.readable_table(agreement.query("model=='aviris_fsca'"), {"flag":"雪标志", "roi":"区域",
    "threshold":"覆盖判雪门槛", "n":"单元数", "tp":"TP", "fn":"FN", "fp":"FP", "tn":"TN", "recall":"参照召回率", "specificity":"参照特异度"}))
print("真实17个单元均为阴性：召回率分母为0，按定义不计算，表中为—。")

## 5 独立覆盖参照需要区分模型分支

WorldView 公开派生产品给出可见雪覆盖面积比例。2016 年的 Landsat 与参考同日；2017 年的 Landsat 比参考晚一天。六波段是 B2–B7，四波段是 B3/B5/B6/B7；固定 200 μm 和多粒径各有一套，历史分支共四方法。

先重算历史归档的共同域分层统计。两景、两种波段、四种分层形成 16 组固定与多粒径比较：多粒径 RMSE 较小 5 组、较大 11 组。2017 年总体被无雪样本主导，混合雪误差需单独阅读。这些统计只评价历史库谱分支，不用于推断 2013 年影像背景模型的绝对精度。

In [ ]:
worldview = t.worldview_metrics()
frozen = pd.read_csv(demo/"stratified_metrics.csv")
comparison = worldview.merge(frozen[["scene","method","stratum","common_n","common_rmse"]],
    on=["scene","method","stratum"], validate="one_to_one")
np.testing.assert_array_equal(comparison.n, comparison.common_n)
np.testing.assert_allclose(comparison.rmse, comparison.common_rmse, atol=5e-7, equal_nan=True)
historical_pairs = worldview.pivot(index=["scene","stratum"], columns="method", values="rmse")
wins = sum(int((historical_pairs[f"multi_{band}"] < historical_pairs[f"fixed200_{band}"]).sum())
    for band in ("six_bands","four_bands"))
assert wins == 5
historical_view = worldview.query("method in ['fixed200_six_bands','multi_six_bands'] and stratum in ['all','mixed']").copy()
historical_view["rmse_pp"] = 100*historical_view.rmse
display(t.readable_table(historical_view, {"scene":"场景", "method":"历史方法", "stratum":"覆盖分层", "n":"共同像元", "rmse_pp":"覆盖 RMSE / pp"}))
figure = t.plot_worldview_strata(worldview)
display(figure); plt.close(figure)

### 用公开端元重新比较 MGS 与 FCLS

接着从观测重新解混，不读取历史预测系数。输入是两景各 12,000 条 B2–B7 反射率、角度匹配 SNICAR 雪库以及公开 USGS 松针和石英谱。参考标签只在解混结束后评价，未用于挑选端元。这个新分支与 2013 年的影像背景实验、历史花岗岩背景分支分别命名。

半径固定/多粒径、六/四波段、MGS/FCLS 组合为八方法。先取八方法共同可归一化像元，再按参考雪比例分层：<0.1 无雪，0.1–0.9 混合雪，≥0.9 近纯雪。配对区间按原始网格 1.5 km 区块重采样 1,000 次；少于 5 个区块时只给点估计，区间显示—。

In [ ]:
with np.load(demo/"transfer_library.npz", allow_pickle=False) as archive:
    library = dict(archive)
transfer, paired_transfer, predictions = independent_transfer(demo, library)
frozen_transfer = pd.read_csv(demo/"transfer_metrics.csv")
np.testing.assert_array_equal(transfer.n, frozen_transfer.n)
np.testing.assert_allclose(transfer.select_dtypes("number"), frozen_transfer.select_dtypes("number"),
    atol=1e-7, equal_nan=True)
view = transfer.query("stratum in ['all','mixed'] and method in ['fixed200_six_MGS','multi_six_MGS','fixed200_six_FCLS','multi_six_FCLS']")
display(t.readable_table(view, {"scene":"场景", "method":"公开库谱方法", "stratum":"覆盖分层", "n":"共同像元",
    "rmse_pp":"覆盖 RMSE / pp", "spectral_rms_pp":"光谱 RMS / pp", "feasible_fraction":"全部系数可行比例"}))
figure = t.plot_transfer_comparison(transfer)
display(figure); plt.close(figure)
delta = paired_transfer[paired_transfer.stratum.isin(["all","mixed"]) & paired_transfer["first"].str.contains("six")].copy()
for column in ("difference","ci_low","ci_high"):
    delta[column] = 100*delta[column]
display(t.readable_table(delta, {"scene":"场景", "stratum":"覆盖分层", "first":"基准方法",
    "second":"比较方法", "n":"像元", "spatial_blocks":"空间块", "difference":"RMSE差 / pp",
    "ci_low":"95%下限 / pp", "ci_high":"95%上限 / pp"}))

在这组公开库谱试验中，FCLS 得到可行系数，并降低了两景混合雪误差；多粒径的改善仍依赖场景和波段。尤其在 MGS 中，增加可选雪谱能继续降低光谱 RMS，却可能扩大覆盖误差。约束、谱库完整性与模型选择分别起作用，最终用相同独立参照比较它们。

完整分层、气溶胶和质量字段保留在 CSV 及本页计算结果中。2017 年一天时间差、山地地形与可见雪面积定义，属于这个外部实验的具体条件。

### 改用具同日独立参考的 MODIS 场景

2013 年严格集合缺少雪阳性，所以新的验证直接使用 2016-02-13 和 2017-05-21 的 MOD09GA，与同日公开 WorldView 派生覆盖配对。先将作者发布的 30 m 覆盖按真实相交面积聚合到 MODIS 的 463.313 m 网格，要求参考有效面积≥95%。商业原始影像没有参与。

这条新模型分支用 NumPy WW80 雪谱和 USGS 松针、石英谱。太阳角采用原生产品在参考支持域内的中位值，分别为 53.23° 和 23.01°；2016 年域内角度变化较大，这一固定角度近似与观测几何差异一并说明。严格七波段 QA 与原来的主规则相同；另列预先指定的五波段对照，保留 1/2/4/6/7 波段的最高质量，并要求晴空且无邻云。

这里把两个问题分开：500 m 反射率解混评价覆盖误差；1 km 原生状态标志与独立覆盖参照比较分类表现。后者只需状态 QA 与参考面积支持，不附加七波段反射率要求。每种波段配置中的四方法共用同一可归一化集合，各级数量如下。

日期在 CSV 中形似整数，读取时明确指定为字符串，用作空间单元标识；数值断言比较覆盖与误差指标。

In [ ]:
from snowmix.modis_validation import evaluate_pixels, evaluate_flags
modis_pixels = pd.read_csv(demo/"modis_reference_pixels.csv.gz", dtype={"scene":str})
modis_units = pd.read_csv(demo/"modis_reference_1km.csv", dtype={"scene":str})
with np.load(demo/"modis_external_library.npz", allow_pickle=False) as archive:
    modis_library = dict(archive)
modis_metrics, modis_delta, modis_support, modis_predictions = evaluate_pixels(modis_pixels, modis_library)
frozen = pd.read_csv(demo/"modis_external_metrics.csv", dtype={"scene":str})
np.testing.assert_array_equal(modis_metrics.n, frozen.n)
np.testing.assert_allclose(modis_metrics.select_dtypes("number"), frozen.select_dtypes("number"),
    atol=1e-7, equal_nan=True)
strict_support = modis_support.query("tier=='strict7'")
np.testing.assert_array_equal(strict_support.common_n, [1416,1817])
np.testing.assert_array_equal(strict_support.reference_positive_ge_0p1, [463,22])
display(t.readable_table(modis_support, {"scene":"日期", "tier":"波段QA", "reference_supported_n":"参考面积合格",
    "qa_supported_n":"再经QA", "common_n":"共同可归一化", "no_snow_n":"无雪", "mixed_n":"混合雪", "full_snow_n":"近纯雪"}, precision=0))
view = modis_metrics.query("tier=='strict7' and stratum in ['all','mixed']")
display(t.readable_table(view, {"scene":"日期", "method":"方法", "stratum":"覆盖分层", "n":"单元数",
    "rmse_pp":"覆盖 RMSE / pp", "spectral_rms_pp":"光谱 RMS / pp"}))
figure = t.plot_modis_external(modis_metrics)
display(figure); plt.close(figure)

cover_bins = modis_metrics[(modis_metrics.scene.astype(str)=="20160213") & (modis_metrics.tier=="strict7") & (modis_metrics.method=="multi_FCLS") & modis_metrics.stratum.isin(["no_snow","low_mixed","snow_dominated"])]
display(t.readable_table(cover_bins, {"stratum":"覆盖水平", "n":"单元数", "bias_pp":"bias / pp", "rmse_pp":"RMSE / pp"}))

2016 年四方法共同域从 QA 后 1,471 个单元降到 1,416 个，排除的 55 个均为参考无雪；2017 年排除的 8 个也均为无雪。三类覆盖数量直接展示筛选后的结构。2017 年只有 22 个混合雪单元，总体误差受 1,795 个无雪单元主导。

随后在唯一 1 km 单元上，以参考雪比例≥0.1 作为主判雪门槛，0.5 和 0.9 检查定义敏感性。2016 年状态 QA 后有 408 个单元，123 个参考阳性；bit15 的 TP/FN/FP/TN 为 74/49/1/284，召回率约 0.602。2017 年只有 5 个阳性，不能仅依据其 100% 点估计评价总体雪检测性能。

在预定的 0.1 与 0.5 阈值下，还可构成三个互斥的覆盖水平：无雪（<0.1）、低至中等雪比例（0.1–0.5）、雪占优势（≥0.5）。2016 年分别有 953、344、119 个单元。原来的近纯雪（≥0.9）层仍保留，只有 10 个，不将它与雪占优势混称。

In [ ]:
flags = evaluate_flags(modis_units)
frozen_flags = pd.read_csv(demo/"modis_external_flags.csv", dtype={"scene":str})
np.testing.assert_allclose(flags.select_dtypes("number"), frozen_flags.select_dtypes("number"), atol=1e-10, equal_nan=True)
primary_flags = flags.query("reference_threshold==.1")
display(t.readable_table(primary_flags, {"scene":"日期", "bit":"状态位", "n":"原生单元",
    "reference_positive":"参照阳性", "reference_negative":"参照阴性",
    "tp":"TP", "fn":"FN", "fp":"FP", "tn":"TN", "recall":"召回率", "specificity":"特异度"}))
# 配对区间保持同一空间区块抽样；差值定义为第二方法减第一方法。
comparison = modis_delta.query("tier=='strict7' and stratum in ['all','mixed']").copy()
for column in ("difference","ci_low","ci_high"):
    comparison[column] *= 100
display(t.readable_table(comparison, {"scene":"日期", "stratum":"覆盖分层", "first":"基准",
    "second":"比较", "n":"单元数", "difference":"RMSE差 / pp", "ci_low":"95%下限 / pp", "ci_high":"95%上限 / pp"}))

同日新数据补回了可评价的雪阳性，而结果仍具有场景依赖：2016 年严格混合雪层中，多粒径 FCLS 的覆盖 RMSE 略高于固定 200 μm；2017 年多粒径较低，但混合雪样本只有 22 个。各数据分支都显示，光谱拟合、系数可行性和独立覆盖误差需要分别阅读。

输入、比例因子、掩膜、精确足迹聚合、角度近似和来源哈希见数据说明。这里的独立检验评价这条 WW80/USGS 模型分支，与 2013 年影像背景模型分别表述。

## 6 尺度处理的四个对应关系

固定字典控制分开比较仅光谱退化、仅空间聚合、二者同时进行，以及“先解混后聚合”与“先聚合后解混”的顺序。仅光谱一行在 30 m 的 62,870 个共同像元上计算；其余行使用粗网格，并分别要求参照的有效面积≥95%。参照值是高分辨率模型输出，不是实测雪覆盖。

这些结果来自已归档的历史固定字典输出；这里逐值重算差异统计，完整影像流程与当前三模型重建的复现级别在首页分开列出。

In [ ]:
with np.load(demo/"historical_spectral_control.npz", allow_pickle=False) as archive:
    spectral_reference, spectral_only = archive["reference"], archive["spectral_only"]
assert len(spectral_reference) == 62870
rows = [{"comparison":"仅光谱退化（30 m）", **t.difference_metrics(spectral_reference, spectral_only)}]
control = pd.read_csv(demo/"same_dictionary_control_corrected.csv")
reference_valid = (control.reference_valid_area_fraction>=.95) & (control.coverage>=.95)
order_valid = (control.spectral_reference_valid_area_fraction>=.95) & (control.coverage>=.95)
for name,column in (("仅空间聚合","spatial_only_snow"),("光谱和空间同时退化","combined_snow")):
    rows.append({"comparison":name, **t.difference_metrics(control.loc[reference_valid,"reference_snow"], control.loc[reference_valid,column])})
rows.append({"comparison":"处理顺序差（同一七波段）", **t.difference_metrics(
    control.loc[order_valid,"fine_modis_aggregated_snow"], control.loc[order_valid,"combined_snow"])})
scale_table = pd.DataFrame(rows)
for column in ("bias","mae","rmse"):
    scale_table[column] *= 100
display(t.readable_table(scale_table, {"comparison":"对照", "n":"共同样本数", "bias":"bias / pp",
    "mae":"MAE / pp", "rmse":"RMSE / pp"}))

## 回到研究问题

固定半径扫描量化雪端元改变如何影响同一观测；有符号系数和 FCLS 对照帮助判断哪些估计能作为面积解释；MODIS 空间筛选定位验证缺失的类别；独立覆盖参照检验光谱拟合改善是否转化为覆盖误差改善。

练习可另开 cell：改变失配条件或预设的覆盖判雪门槛，先预测哪个误差会变化，再重算。在比较方法前重新检查共同域，记录本次输入、输出物理量、样本数和图的变化。保留基线断言，练习使用独立变量。

WorldView 派生雪覆盖参考来自 Stillinger 与 Bair，Zenodo 4031446，CC BY 4.0；已裁剪、聚合和分层。实验采用公开 NASA/JPL、USGS 与授权开放资料，商业原始 WorldView 影像未纳入。